# Day 13 Observathon - Colab Runner

Base path dùng chung: `/content/2A202600883-MaiHanhPham-Day-13-Lab-Observathon`

Checklist từ README: sửa `solution/config.json`, `solution/prompt.txt`, `solution/wrapper.py`, `solution/findings.json`; chạy `harness/selfcheck.py`; chạy simulator; tạo `run_output.json`, `score.json` nếu có scorer; zip/gửi `solution/` cùng output.

## 0. Chuẩn bị project

Upload hoặc clone thư mục project sao cho tồn tại đúng path bên dưới. Nếu đang dùng Google Drive, mount Drive trước rồi copy thư mục vào `/content`.

In [ ]:
from pathlib import Path
import os, json, glob, subprocess, zipfile

BASE = Path('/content/2A202600883-MaiHanhPham-Day-13-Lab-Observathon')
if not BASE.exists():
    raise FileNotFoundError(f'Không thấy project tại {BASE}. Hãy upload/copy đúng thư mục vào /content.')
os.chdir(BASE)
print('Working directory:', Path.cwd())
print('Files:', [p.name for p in BASE.iterdir()][:20])

## 1. Set API key

Nếu dùng OpenAI, nhập key ở cell dưới. Nếu dùng local OpenAI-compatible endpoint, sửa `solution/config.json` sang provider local và set `LOCAL_BASE_URL`.

In [ ]:
import getpass
if not os.environ.get('OPENAI_API_KEY'):
    key = getpass.getpass('OPENAI_API_KEY: ')
    if key:
        os.environ['OPENAI_API_KEY'] = key
print('OPENAI_API_KEY set:', bool(os.environ.get('OPENAI_API_KEY')))

## 2. Selfcheck

In [ ]:
!python harness/selfcheck.py

## 3. Tìm binary chạy trên Colab

Colab là Linux nên không chạy được `observathon-sim.exe`. Cần có binary Linux ở `bin/practice/observathon-sim` hoặc upload/gắn đúng gói Linux rồi giải nén vào project.

In [ ]:
candidates = [Path(p) for p in glob.glob('bin/practice/observathon-sim*')]
candidates += [Path(p) for p in glob.glob('observathon-sim/observathon-sim')]
candidates += [Path(p) for p in glob.glob('observathon-sim/observathon-sim.exe')]
print('Candidates:', [str(p) for p in candidates])

SIM = None
for p in candidates:
    if p.name.endswith('.exe'):
        print('Skip Windows exe on Colab:', p)
        continue
    SIM = p
    break

if SIM is None:
    raise FileNotFoundError('Chưa có Linux simulator. Upload bản Linux vào bin/practice/observathon-sim rồi chạy lại cell này.')

SIM.chmod(0o755)
print('Using simulator:', SIM)

## 4. Chạy practice sim

In [ ]:
cmd = [str(SIM), '--config', 'solution/config.json', '--wrapper', 'solution/wrapper.py', '--out', 'run_output.json', '--users', '80', '--turns', '8', '--concurrency', '8']
print(' '.join(cmd))
subprocess.run(cmd, check=True)
data = json.loads(Path('run_output.json').read_text(encoding='utf-8'))
print('Top-level keys:', list(data.keys()))
print('Rows:', len(data.get('results', [])))
print(json.dumps(data.get('results', [])[:3], ensure_ascii=False, indent=2))

## 5. Xem telemetry wrapper

In [ ]:
log_files = sorted(Path('logs').glob('*.log')) if Path('logs').exists() else []
print('Log files:', [str(p) for p in log_files])
if log_files:
    rows = []
    for line in log_files[-1].read_text(encoding='utf-8').splitlines():
        try:
            rows.append(json.loads(line))
        except Exception:
            pass
    calls = [r['data'] for r in rows if r.get('event') == 'AGENT_CALL']
    print('AGENT_CALL count:', len(calls))
    if calls:
        statuses = {}
        total_cost = 0.0
        for c in calls:
            statuses[c.get('status')] = statuses.get(c.get('status'), 0) + 1
            total_cost += float(c.get('cost_usd') or 0)
        print('Statuses:', statuses)
        print('Estimated wrapper cost USD:', round(total_cost, 6))
        print(json.dumps(calls[:3], ensure_ascii=False, indent=2))

## 6. Chạy scorer nếu đã có binary

In [ ]:
score_candidates = [Path(p) for p in glob.glob('bin/practice/observathon-score*')]
score_candidates += [Path(p) for p in glob.glob('observathon-score/observathon-score')]
SCORER = next((p for p in score_candidates if not p.name.endswith('.exe')), None)
if SCORER:
    SCORER.chmod(0o755)
    cmd = [str(SCORER), '--run', 'run_output.json', '--out', 'score.json']
    print(' '.join(cmd))
    subprocess.run(cmd, check=True)
    print(Path('score.json').read_text(encoding='utf-8')[:2000])
else:
    print('Không tìm thấy Linux scorer; bỏ qua bước score.json.')

## 7. Zip file nộp

In [ ]:
submit_files = ['solution/config.json', 'solution/prompt.txt', 'solution/examples.json', 'solution/wrapper.py', 'solution/findings.json']
for extra in ['run_output.json', 'score.json']:
    if Path(extra).exists():
        submit_files.append(extra)

zip_path = BASE / 'observathon_submission.zip'
if zip_path.exists():
    zip_path.unlink()
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in submit_files:
        z.write(f)
print('Created:', zip_path)
print('Included:', submit_files)